# BiCyc Multi-Adapter — Huấn luyện trên Google Colab

Chạy pipeline **KeepLoRA + BiCyc + PFD routing** (Hướng 1) trên GPU Colab thay vì máy local:
- GPU miễn phí: **T4 16GB** (khuyên dùng, hỗ trợ fp16 TensorCore).
- Bật **AMP fp16** + TF32 để tăng tốc ~1.5–2x.
- CIFAR-100 tự tải lần chạy đầu.
- Tuỳ chọn lưu dữ liệu/kết quả vào **Google Drive** để không mất khi phiên reset.

**Chuẩn bị code — chọn 1 trong 3 cách:**
- **Cách A (khuyên dùng):** điền `REPO_URL` (URL GitHub) ở cell dưới — Colab có Internet sẵn.
- **Cách B:** upload repo lên Drive, mount Drive rồi điền `DRIVE_REPO`.
- **Cách C:** kéo-thả file zip repo vào khung Files của Colab và giải nén tại `/content`.

**Chọn GPU:** *Runtime → Change runtime type → T4 GPU*.

In [ ]:
# 1) Kiểm tra GPU
!nvidia-smi -L || echo "CANH BAO: chua bat GPU! Runtime -> Change runtime type -> T4 GPU"
import torch
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())

In [ ]:
# 2) (Tuỳ chọn) Mount Google Drive để giữ data + kết quả qua các phiên
USE_DRIVE = True   # đặt False nếu chỉ chạy thử nhanh
DRIVE_MOUNTED = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_MOUNTED = True
print("Drive:", DRIVE_MOUNTED)

In [ ]:
# 3) Lấy code về /content/repo (Cách A: git clone | Cách B: Drive)
import shutil
from pathlib import Path

REPO_URL = ""    # Cách A: vd "https://github.com/<user>/BiCyc_MultiAdapter.git"
DRIVE_REPO = ""  # Cách B: vd "/content/drive/MyDrive/BiCyc_MultiAdapter"

WORK_REPO = Path("/content/repo")
candidates = [Path(p) for p in [DRIVE_REPO] if p]
repo = next((c for c in candidates if (c / "pyproject.toml").exists()), None)
if repo is None and REPO_URL:
    import subprocess
    if WORK_REPO.exists():
        shutil.rmtree(WORK_REPO)
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(WORK_REPO)], check=True)
    repo = WORK_REPO
assert repo is not None, "Dien REPO_URL hoac DRIVE_REPO o cell tren (hoac tai zip repo vao /content)."
if repo.resolve() != WORK_REPO.resolve():
    if WORK_REPO.exists():
        shutil.rmtree(WORK_REPO)
    shutil.copytree(repo, WORK_REPO, ignore=shutil.ignore_patterns(".git", "__pycache__", ".venv"))
print("Repo san sang:", WORK_REPO)

In [ ]:
# 4) Cài dependencies (torch/torchvision đã có sẵn trên Colab; base.txt không đụng tới torch)
%pip install -q -r {WORK_REPO}/requirements/base.txt
%pip install -q -e {WORK_REPO}
import bicyc_multiadapter
print("bicyc_multiadapter", bicyc_multiadapter.__version__)

## Cấu hình thí nghiệm

| Preset | Khi nào dùng |
| --- | --- |
| `keeplora_bicyc` | Đề xuất đầy đủ (routed multi-adapter + BiCyc + adaptive gate), batch 128 |
| `keeplora_bicyc_8gb` | Batch 32 + AMP bật sẵn |
| `keeplora_original` | Baseline KeepLoRA nguyên gốc (merge sau task) |

Với T4 16GB: `keeplora_bicyc` + `BATCH_SIZE = 128` + `USE_AMP = True`.

> ⚠️ ViT-B/16 @224 × 20 epochs × 10 tasks **rất lâu** (hàng giờ đến >12h). Đặt
> `EPOCHS_PER_TASK = 1` chạy thử cả pipeline trước. Colab free giới hạn phiên ~4–6h và có thể
> ngắt bất kỳ lúc nào — hãy dùng `USE_DRIVE = True` để dữ liệu/kết quả không mất; nếu dở dang,
> tải `checkpoint_last.pt` về và đánh giá local bằng `python -m bicyc_multiadapter.evaluate`.

In [ ]:
# 5) Tham số run — sửa ở đây thay vì sửa yaml
EXPERIMENT = "keeplora_bicyc"      # keeplora_bicyc | keeplora_bicyc_8gb | keeplora_original | keeplora_original_8gb
SEED = 2024
EPOCHS_PER_TASK = 20               # đặt 1 để smoke test nhanh toàn bộ pipeline
BATCH_SIZE = None                  # None = giữ theo preset; T4 16GB có thể để 128
USE_AMP = True                     # fp16 mixed precision (hiệu quả nhất trên T4)
NUM_WORKERS = 2                    # Colab có 2 vCPU
ACTIVATION_CACHE_ROWS = 4096       # giảm xuống 1536 nếu hết RAM
CHECKPOINT_EVERY_EPOCHS = 1        # lưu snapshot mỗi N epoch để resume khi Colab ngắt (ghi đè, ~350MB)

_base = "/content/drive/MyDrive/bicyc" if DRIVE_MOUNTED else "/content"
DATA_ROOT = f"{_base}/data/cifar100"
OUT_DIR = f"{_base}/outputs/{EXPERIMENT}/seed_{SEED}"

overrides = [
    f"experiment={EXPERIMENT}",
    f"experiment.seed={SEED}",
    f"data.root={DATA_ROOT}",
    f"data.num_workers={NUM_WORKERS}",
    f"output_dir={OUT_DIR}",
    f"experiment.activation_cache_rows={ACTIVATION_CACHE_ROWS}",
    f"experiment.train.amp={str(USE_AMP).lower()}",
    f"experiment.checkpoint_every_epochs={CHECKPOINT_EVERY_EPOCHS}",
]
if EPOCHS_PER_TASK is not None:
    overrides.append(f"experiment.train.epochs_per_task={EPOCHS_PER_TASK}")
if BATCH_SIZE is not None:
    overrides.append(f"experiment.train.batch_size={BATCH_SIZE}")
print("Hydra overrides:\n  " + "\n  ".join(overrides))

In [ ]:
# 6) HUẤN LUYỆN — cell này chứa cả 10 tasks; tqdm log từng epoch
import subprocess

cmd = ["python", "-m", "bicyc_multiadapter.train", *overrides]
print("$", " ".join(cmd), "\n")
result = subprocess.run(cmd, cwd=WORK_REPO)
assert result.returncode == 0, "Training that bai — xem log phia tren."

In [ ]:
# 7) (Tuỳ chọn) Theo dõi loss/metric trực tiếp bằng TensorBoard inline
%load_ext tensorboard
%tensorboard --logdir {OUT_DIR}/tensorboard

## Đánh giá & trực quan kết quả

In [ ]:
# 8) Đọc metrics và vẽ accuracy matrix
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

out_dir = Path(OUT_DIR)
metrics = json.loads((out_dir / "metrics.json").read_text())
print("=== Summary ===")
print(json.dumps(metrics["summary"], indent=2))

matrix = np.array(metrics["accuracy_matrix"], dtype=float)
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(matrix, cmap="viridis", vmin=0, vmax=1)
for i in range(matrix.shape[0]):
    for j in range(matrix.shape[1]):
        if not np.isnan(matrix[i, j]):
            ax.text(j, i, f"{matrix[i, j]:.2f}", ha="center", va="center", color="w", fontsize=8)
ax.set_xlabel("Task da thay (eval)")
ax.set_ylabel("Sau khi hoc task")
summary = metrics["summary"]
ax.set_title(
    f"{EXPERIMENT} seed={SEED} - last={summary['last_average']:.3f}, forget={summary['forgetting']:.3f}"
)
fig.colorbar(im, ax=ax, label="Accuracy")
plt.tight_layout()
plt.savefig(out_dir / "accuracy_matrix.png", dpi=150)
plt.show()

In [ ]:
# 9) Lưu kết quả vào Drive (nếu mount) và tải zip về máy
import shutil
from pathlib import Path

archive = shutil.make_archive("/content/results", "zip", out_dir)
if DRIVE_MOUNTED:
    dest = Path("/content/drive/MyDrive/bicyc_results")
    dest.mkdir(parents=True, exist_ok=True)
    shutil.copy2(archive, dest / Path(archive).name)
    print("Da copy sang Drive:", dest / Path(archive).name)
from google.colab import files
files.download(archive)

## Mẹo vận hành trên Colab

- **T4** hỗ trợ fp16 TensorCore → `USE_AMP = True` lợi ích lớn nhất; Colab Pro có A100/V100 hỗ trợ thêm bf16 (`experiment.train.amp_dtype=bfloat16`).
- Ablation nhanh bằng override trong `overrides`, ví dụ:
  - Tắt adaptive gate (λ_t ≡ 1): `experiment.alignment.adaptive_gate=false`
  - Baseline nguyên gốc: `experiment=keeplora_original`
  - Merge thay vì routed: `experiment.keeplora.merge_after_task=true`
  - Đổi seed: `experiment.seed=0` (chạy ≥ 3 seeds mỗi cấu hình khi báo cáo).
- Colab free **ngắt phiên bất định (~4–6h)**: luôn `USE_DRIVE = True`; data CIFAR-100 trong
  `Drive/bicyc/data/` được tái sử dụng nên không tải lại.
- **Tự động resume**: pipeline lưu `checkpoint_boundary.pt` (sau mỗi task) và `checkpoint_live.pt`
  (mỗi `CHECKPOINT_EVERY_EPOCHS` epoch hoặc khi interrupt). Colab ngắt phiên? Chỉ cần chạy lại cell
  huấn luyện với cùng `OUT_DIR` — training tiếp tục từ đúng epoch giữa task, kèm optimizer + RNG
  state. File checkpoint luôn ghi đè nên chỉ tốn dung lượng của 1 bản (~350MB với ViT-B).
- So sánh catastrophic forgetting: `history.jsonl` (mỗi task một dòng: per-task accuracy + tích lũy
  last/incremental/forgetting), TensorBoard scalar `eval/running_forgetting`, loss theo epoch trong
  `train_log.csv`.
- Muốn đánh giá lại không train: `python -m bicyc_multiadapter.evaluate experiment=<name>
  output_dir=<OUT_DIR> data.root=<DATA_ROOT>`.